# Trajectory annotation

**Linking every trajectory point to the conditions the moving object passed through.**

Movement data and context data are observed at different resolutions, and movement is far denser. AIS messages arrive within minutes; weather is reported hourly, from a handful of coastal stations. So a vessel's position at 14:37, halfway between two stations, has no wind measurement of its own.

Annotation supplies one, in two stages:

| | | |
|---|---|---|
| **1. Spatial** | across stations, one epoch at a time | *what was the wind here, at that hour?* |
| **2. Temporal** | through time, at that position | *what was the wind here, at that minute?* |

Each stage has candidate methods, and each is compared by cross-validation.

The method applies to any moving object — vessels, aircraft, vehicles, wildlife, pedestrians — and to any context measured at fixed stations. This notebook works through it on real AIS and weather data from the western English Channel and Bay of Biscay, February 2016: a 24-hour sample that ships with the repository, or the full public archive it comes from (section 1); the maritime data is one application, not the scope. Every plot is interactive &mdash; hover for values, zoom, and click legend entries to mute a series. They need a live kernel, so they appear blank in a static render such as GitHub's notebook preview.

---

## Setup

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

import hvplot.pandas  # noqa: F401  -- adds .hvplot to DataFrames
import holoviews as hv

hv.extension("bokeh")

import warnings
from sklearn.exceptions import ConvergenceWarning
# Kriging's optimiser warns when a dozen stations cannot pin its length scale
# down; the fit is still made and scored, so the warning is only noise here.
warnings.filterwarnings("ignore", category=ConvergenceWarning)

REPO = Path.cwd().parent
DATA = REPO / "data" / "sample"

from catraj.annotation import (
    Nearest, IDW, TIN, CloughTocher, RBF, Kriging, to_surface,
    DLT, PiecewisePolynomial, Spline,
    evaluate_spatial, evaluate_temporal,
    annotate, annotate_points, decode_compass,
    parse_data_dictionary, describe_columns,
)

# Every table, figure and annotated track this notebook produces is written
# to the repository's results folder, so results outlive the kernel session.
RESULTS = REPO / "results" / "01_trajectory_annotation"
RESULTS.mkdir(parents=True, exist_ok=True)


def save(obj, name, note=""):
    """Write a result to RESULTS, choosing the format from its type.

    DataFrames become CSV, GeoDataFrames GeoPackage, and holoviews objects
    standalone HTML that opens in any browser with no kernel and no network.
    """
    if isinstance(obj, gpd.GeoDataFrame):
        path = RESULTS / (name + ".gpkg")
        obj.to_file(path, driver="GPKG")
    elif isinstance(obj, pd.DataFrame):
        path = RESULTS / (name + ".csv")
        obj.to_csv(path, index=False)
    elif isinstance(obj, pd.Series):
        path = RESULTS / (name + ".csv")
        obj.to_frame().to_csv(path)
    elif isinstance(obj, str):
        path = RESULTS / (name + ".txt")
        path.write_text(obj, encoding="utf-8")
    else:                                    # holoviews / bokeh figure
        path = RESULTS / (name + ".html")
        hv.save(obj, path, backend="bokeh")
    print(f"  saved  {path.name:<42} {path.stat().st_size / 1024:8.1f} KB  {note}")
    return obj


# ---- column names ---------------------------------------------------------------
# The sources use coded names (Ff, Td, id_windDir, sourcemmsi, ...). On loading,
# every column is renamed to what it measures, following each archive's own
# field descriptions (section 1 shows them side by side). Shapefiles truncate
# names to ten characters, so both spellings are listed.
WEATHER_NAMES = {
    "id_station": "station_id", "local_time": "unix_time",
    "T": "air_temperature", "Tn": "min_air_temperature", "Tx": "max_air_temperature",
    "P": "air_pressure", "U": "relative_humidity", "Td": "dew_point",
    "id_windDirection": "wind_direction_code", "id_windDir": "wind_direction_code",
    "Ff": "wind_speed", "ff10": "max_gust_10min", "ff3": "max_gust", "VV": "visibility",
    "RRR": "precipitation", "tR": "precipitation_period",
}
AIS_NAMES = {
    "sourcemmsi": "mmsi", "t": "unix_time",
    "navigationalstatus": "navigational_status", "navigation": "navigational_status",
    "rateofturn": "rate_of_turn",
    "speedoverground": "speed_over_ground", "speedoverg": "speed_over_ground",
    "courseoverground": "course_over_ground", "courseover": "course_over_ground",
    "trueheading": "true_heading", "trueheadin": "true_heading",
    "vesselType": "vessel_type", "shiptype": "ship_type_code",
}

# ---- data source --------------------------------------------------------------
# "sample": the 24-hour extract shipped in data/sample/ -- no download needed
# "zenodo": the full public archive, in data/zenodo_1167595/ (see section 1)
# "auto"  : zenodo when it has been downloaded, otherwise the sample
SOURCE = "auto"
WINDOW = ("2015-10-01", "2016-04-01")   # zenodo: the period (UTC); this is the whole archive
MAX_VESSELS = None                      # zenodo: None for every vessel, or the N busiest
DOWNLOAD = False                        # zenodo: True fetches the files below (~200 MB)
N_JOBS = -1                             # processes for the heavy steps: -1 = every core, 1 = serial

ZENODO = REPO / "data" / "zenodo_1167595"
ZENODO_FILES = {   # Zenodo name -> (md5 from the record, members, target folder)
    "[P1] AIS Data.zip": ("90bec666a3c3474e18bb57676348b655",
                          ["nari_dynamic.csv", "nari_static.csv", "README_parameters.txt"],
                          ZENODO / "P1_AIS_Data"),
    "[E2] Weather Conditions.zip": ("d8ac0644a670f584daaa4a450cf2baa7",
                                    ["table_wheatherObservation.csv", "table_weatherStation.csv",
                                     "README_parameters.txt"], ZENODO / "E2_Weather_Conditions"),
    "[P1] AIS Status, Codes and Types.zip": ("6f9bb328d6e657b16fea55210a6e56cf", ["Ship Types List.csv"], ZENODO / "P1_AIS_Codes"),
}


def fetch(name, md5, members, target):
    """Download one archive from the Zenodo record, check it, extract what is needed."""
    import hashlib, urllib.request, zipfile
    zpath = ZENODO / (name.replace("[", "").replace("] ", "_").replace(",", "")
                      .replace(" ", "_"))
    if not zpath.exists():
        url = ("https://zenodo.org/records/1167595/files/"
               + urllib.request.quote(name) + "?download=1")
        print(f"downloading {name} ...")
        urllib.request.urlretrieve(url, zpath)
    if md5 and hashlib.md5(zpath.read_bytes()).hexdigest() != md5:
        raise RuntimeError(f"{zpath.name}: checksum mismatch -- delete it and run again")
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zpath) as z:
        for m in members:
            if not (target / m).exists():
                z.extract(m, target)


if DOWNLOAD:
    ZENODO.mkdir(parents=True, exist_ok=True)
    for name, spec in ZENODO_FILES.items():
        fetch(name, *spec)

downloaded = all((target / m).exists() for _, members, target in ZENODO_FILES.values()
                 for m in members)
if SOURCE == "auto":
    SOURCE = "zenodo" if downloaded else "sample"
elif SOURCE == "zenodo" and not downloaded:
    raise FileNotFoundError("the Zenodo files are missing: set DOWNLOAD = True above, or run\n"
                            "  python scripts/download_data.py annotation\n"
                            "(see docs/getting_the_data.md)")
print(f"data source: {SOURCE}"
      + (f"  ({WINDOW[0]} to {WINDOW[1]}, "
         + ("every vessel)" if MAX_VESSELS is None else f"the {MAX_VESSELS} busiest vessels)")
         if SOURCE == "zenodo" else ""))
print(f"ready - results go to {RESULTS.relative_to(REPO).as_posix()}/")

data source: zenodo  (2015-10-01 to 2016-04-01, every vessel)
ready - results go to results_trajectory_annotation/


Results are written to **`results/01_trajectory_annotation/`** in the repository.
Each file is named for the step that produced it, so the folder reads as a record of the
analysis:

| prefix | what it holds |
|---|---|
| `01_...` | the input data, as loaded |
| `02_...` | stage 1 &mdash; spatial interpolation and its cross-validation |
| `03_...` | stage 2 &mdash; temporal interpolation and its cross-validation |
| `04_...` | the annotated trajectory |

Figures are saved as **standalone HTML** &mdash; those open in any browser with no kernel
and no network, unlike the plots inline in this notebook.

---
## 1. The data

Two sources, chosen with `SOURCE` in the setup cell. Both are projected to **UTM zone 30N**, so distances are in metres. That matters: spatial interpolation needs a metric space, and degrees are not one &mdash; at 48&deg;N a degree of longitude spans about 74 km against 111 km for a degree of latitude.

**`"sample"`**: two matched shapefiles in `data/sample/`, covering **exactly the same 24 hours**, 2016-02-05 01:00 to 2016-02-06 01:00 UTC:

- **`Weather_Observation_UTM.shp`** &mdash; 16 coastal stations, hourly
- **`AIS_DLT_UTM.shp`** &mdash; 106,894 vessel positions

**`"zenodo"`**: the full public archive the sample was cut from, October 2015 to March 2016, longitude &minus;10&deg; to 0&deg;, latitude 45&deg; to 51&deg;. The notebook takes the period in `WINDOW` and the `MAX_VESSELS` busiest vessels in it.

> Ray, C., Dréo, R., Camossi, E., Jousselme, A.-L., Iphar, C. (2019). *Heterogeneous integrated dataset for Maritime Intelligence, Surveillance, and Reconnaissance.* Data in Brief 25, 104141. [doi:10.1016/j.dib.2019.104141](https://doi.org/10.1016/j.dib.2019.104141) &middot; data: Zenodo [doi:10.5281/zenodo.1167595](https://doi.org/10.5281/zenodo.1167595)

| Zenodo file | Size | Used here | Licence |
|---|---|---|---|
| [`[P1] AIS Data.zip`](https://zenodo.org/records/1167595/files/%5BP1%5D%20AIS%20Data.zip?download=1) | 201 MB | `nari_dynamic.csv` (positions), `nari_static.csv` (vessel type) | **[CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)** |
| [`[E2] Weather Conditions.zip`](https://zenodo.org/records/1167595/files/%5BE2%5D%20Weather%20Conditions.zip?download=1) | 1.3 MB | 16 coastal stations, hourly (rp5.ua, from Met Office / NOAA data) | public, **cite the source** |
| [`[P1] AIS Status, Codes and Types.zip`](https://zenodo.org/records/1167595/files/%5BP1%5D%20AIS%20Status%2C%20Codes%20and%20Types.zip?download=1) | 8 kB | `Ship Types List.csv`, to name vessel types | as the AIS data |

* **AIS, CC BY-NC-SA 4.0**: attribute the Naval Academy Research Institute (NARI), Brest; **no commercial use**; anything derived from it, annotated trajectories included, is shared under the same licence.
* **Weather**: public; cite rp5.ua, the Met Office and NOAA as the source.

**Getting the full archive.** Any one of:

* from a terminal, at the repository root: `python scripts/download_data.py annotation` (checksums verified, resumes if interrupted);
* set `DOWNLOAD = True` in the setup cell and run it;
* download by hand and unzip into `data/zenodo_1167595/P1_AIS_Data/`, `.../E2_Weather_Conditions/` and `.../P1_AIS_Codes/`.

Every step, the expected folder layout, disk space, checksums and troubleshooting are in [**getting_the_data.md**](../docs/getting_the_data.md). `data/zenodo_1167595/` is ignored by git.

Every MMSI is checked against ITU-R M.585 on loading: only ship stations with an allocated flag-state code (MID) are kept, and the counts of everything set aside are shown.

For mapping we also keep a Web Mercator copy, which is what tile basemaps use.

In [2]:
if SOURCE == "sample":
    weather = gpd.read_file(DATA / "Weather" / "Weather_Observation_UTM.shp")
else:
    lo, hi = (pd.Timestamp(s).value // 10**9 for s in WINDOW)
    weather = pd.read_csv(ZENODO / "E2_Weather_Conditions" / "table_wheatherObservation.csv")
    weather = weather[(weather.local_time >= lo - 3600) & (weather.local_time <= hi + 3600)]
    weather = weather.replace(-65536, np.nan)                     # the archive's "missing"
    sites = pd.read_csv(ZENODO / "E2_Weather_Conditions" / "table_weatherStation.csv")
    weather = weather.merge(sites[["id_station", "latitude", "longitude"]], on="id_station")
    weather = gpd.GeoDataFrame(
        weather, geometry=gpd.points_from_xy(weather.longitude, weather.latitude), crs=4326
    ).to_crs(32630).sort_values(["id_station", "local_time"]).reset_index(drop=True)

weather = weather.rename(columns=WEATHER_NAMES)   # coded names -> what they measure
weather["x"] = weather.geometry.x          # UTM metres -- used for interpolation
weather["y"] = weather.geometry.y
weather["time"] = pd.to_datetime(weather.unix_time, unit="s")
weather["station"] = weather.station_id.astype(str)

# Web Mercator copy, for plotting on a basemap
mercator = weather.to_crs(3857)
weather["mx"] = mercator.geometry.x
weather["my"] = mercator.geometry.y

print(f"{len(weather):,} observations · {weather.station.nunique()} stations "
      f"· {weather.unix_time.nunique():,} epochs")
print(f"{weather.time.min()} to {weather.time.max()} UTC · CRS {weather.crs.name}")
save(weather.drop(columns="geometry"), "01_weather_observations",
     f"{weather.station.nunique()} stations x {weather.unix_time.nunique():,} hourly epochs")

weather[["station", "time", "x", "y", "air_temperature", "wind_speed",
         "wind_direction_code", "air_pressure"]].head()

69,810 observations · 16 stations · 4,387 epochs
2015-09-30 23:00:00 to 2016-04-01 01:00:00 UTC · CRS WGS 84 / UTM zone 30N


  saved  01_weather_observations.csv                 11798.9 KB  16 stations x 4,387 hourly epochs


,station,time,x,y,air_temperature,wind_speed,wind_direction_code,air_pressure
0,3803,2015-10-01 00:00:00,263112.557063,5.534591e+06,13.6,7.0,3,772.1
1,3803,2015-10-01 02:00:00,263112.557063,5.534591e+06,13.2,7.0,3,772.0
2,3803,2015-10-01 03:00:00,263112.557063,5.534591e+06,13.2,7.0,3,772.0
3,3803,2015-10-01 04:00:00,263112.557063,5.534591e+06,13.0,7.0,3,771.6
4,3803,2015-10-01 05:00:00,263112.557063,5.534591e+06,12.7,6.0,3,771.6


### Meaningful names

The source columns are coded: `Ff`, `Td`, `id_windDir` say nothing on their own. Both archives ship a plain-text description of their fields, and `parse_data_dictionary` reads it. Here it shows what each column measures, beside the name it is given on loading (`WEATHER_NAMES` and `AIS_NAMES` in the setup cell). From here on the notebook uses those names.

For your own data this is optional: columns whose names already speak for themselves need none of it.

In [3]:
if SOURCE == "sample":
    weather_dict = DATA / "Weather" / "Weather_parameters.txt"
    ais_dict = DATA / "AIS" / "AIS_parameters.txt"
else:
    weather_dict = ZENODO / "E2_Weather_Conditions" / "README_parameters.txt"
    ais_dict = ZENODO / "P1_AIS_Data" / "README_parameters.txt"
weather_fields = parse_data_dictionary(weather_dict)
ais_fields = parse_data_dictionary(ais_dict)

print(f"{len(weather_fields)} weather fields, {len(ais_fields)} AIS fields "
      f"described\n")

# Shapefiles truncate names to ten characters, so `id_windDirection` in the
# dictionary has to match a column called `id_windDir`. describe_columns
# handles that, and falls back to the bare name when nothing is known.
# The AIS field list names some columns differently from the data files, and
# one column of the sample was derived rather than recorded.
AIS_DICT_NAMES = {"navigationalstatus": "status", "navigation": "status",
                  "rateofturn": "turn", "speedoverground": "speed", "speedoverg": "speed",
                  "courseoverground": "course", "courseover": "course",
                  "trueheading": "heading", "trueheadin": "heading"}
DERIVED = {"vesselType": "Vessel type, named from the ship type code of the static messages"}

described = {}
for names, fields, alias in ((WEATHER_NAMES, weather_fields, {}),
                             (AIS_NAMES, ais_fields, AIS_DICT_NAMES)):
    keys = {source: alias.get(source, source) for source in names}
    found = describe_columns(sorted(set(keys.values())), fields)
    described.update({s: found[k].split(" - ", 1)[-1] for s, k in keys.items()})
described.update(DERIVED)

rows, labels = [], {}       # labels: the description, keyed by the name used here
for dataset, names in (("weather", WEATHER_NAMES), ("AIS", AIS_NAMES)):
    listed = set()
    for source, name in names.items():
        meaning = described[source]
        if name in listed or meaning in (source, AIS_DICT_NAMES.get(source)):
            continue        # the other spelling of a column already listed
        listed.add(name)
        labels.setdefault(name, f"{name} - {meaning}")
        rows.append({"data": dataset, "source column": source, "named here": name,
                     "means": meaning})
names_table = pd.DataFrame(rows)
save(names_table, "01_column_names", "source name -> name used here")
names_table

15 weather fields, 24 AIS fields described

  saved  01_column_names.csv                             1.7 KB  source name -> name used here


,data,source column,named here,means
0,weather,id_station,station_id,id of the stations
1,weather,local_time,unix_time,"unix timestamp - number of seconds since ""01/0..."
2,weather,T,air_temperature,Air temperature (°C)
3,weather,Tn,min_air_temperature,Minimum air temperature (°C) (not exceeding 12...
4,weather,Tx,max_air_temperature,Maximum air temperature (°C) (not exceeding 12...
5,weather,P,air_pressure,Atmospheric pressure (mm of mercury)
6,weather,U,relative_humidity,Relative humidity (%)
7,weather,Td,dew_point,Dewpoint temperature (°C)
8,weather,id_windDirection,wind_direction_code,Mean wind direction (compass point)
9,weather,Ff,wind_speed,Mean wind speed (meters per seconds)


In [4]:
if SOURCE == "sample":
    ais = gpd.read_file(DATA / "AIS" / "AIS_DLT_UTM.shp")
else:
    ais = pd.read_csv(ZENODO / "P1_AIS_Data" / "nari_dynamic.csv")
    ais = ais[(ais.t >= lo) & (ais.t < hi)].drop_duplicates(["sourcemmsi", "t"])
    if MAX_VESSELS:
        ais = ais[ais.sourcemmsi.isin(ais.sourcemmsi.value_counts().index[:MAX_VESSELS])]
    # Vessel type: the code from the static messages, named from the archive's list.
    static = (pd.read_csv(ZENODO / "P1_AIS_Data" / "nari_static.csv",
                          usecols=["sourcemmsi", "shiptype"])
                .dropna().drop_duplicates("sourcemmsi", keep="last"))
    codes = pd.read_csv(ZENODO / "P1_AIS_Codes" / "Ship Types List.csv")
    type_names = {c: row.type_name.split(" - ")[0] for row in codes.itertuples()
                  for c in range(row.shiptype_min, row.shiptype_max + 1)}
    ais = ais.merge(static, on="sourcemmsi", how="left")
    ais["vesselType"] = ais.shiptype.map(type_names).fillna("Unknown")
    ais = gpd.GeoDataFrame(ais, geometry=gpd.points_from_xy(ais.lon, ais.lat), crs=4326
                           ).to_crs(32630).sort_values(["sourcemmsi", "t"]).reset_index(drop=True)

ais = ais.rename(columns=AIS_NAMES)               # coded names -> what they measure

# MMSI check (ITU-R M.585). Every MMSI has nine digits; a ship station's is
# MIDxxxxxx, its first three digits an allocated Maritime Identification Digit
# (MID, the flag state). Other patterns belong to coast stations (00MID...),
# groups of ships (0MID...), SAR aircraft (111MID...), aids to navigation
# (99MID...), craft of a parent ship (98MID...) and distress devices
# (970/972/974...). The MIDs below are the ITU's own table
# (itu.int/gladapp/Allocation/MIDs, 2022); the archive's country list is
# incomplete and misses Malta 229, Panama 373-374, Singapore 566 and others.
ITU_MIDS = {m for part in (
    "201-216,218-220,224-259,261-279,301,303-312,314,316,319,321,323,325,327,329-332,"
    "334,336,338-339,341,343,345,347-348,350-359,361-362,364,366-379,401,403,405,408,"
    "410,412-414,416-417,419,422-423,425,428,431-432,434,436-438,440-441,443,445,447,"
    "450-451,453,455,457,459,461,463,466,468,470-473,475,477-478,501,503,506,508,"
    "510-512,514-516,518,520,523,525,529,531,533,536,538,540,542,544,546,548,550,553,"
    "555,557,559,561,563-567,570,572,574,576-578,601,603,605,607-613,615-622,624-627,"
    "629-638,642,644-645,647,649-650,654-657,659-672,674-679,701,710,720,725,730,735,"
    "740,745,750,755,760,765,770,775").split(",")
    for m in range(int(part.split("-")[0]), int(part.split("-")[-1]) + 1)}


def mmsi_class(mmsi):
    """The ITU-R M.585 class of an MMSI, checking its MID wherever it has one."""
    mmsi = int(mmsi)
    if not 0 < mmsi <= 999_999_999:
        return "not nine digits"
    s = f"{mmsi:09d}"                      # stored without its leading zeros
    if len(set(s)) == 1 or s in ("123456789", "012345678"):
        return "placeholder"
    # prefix -> (class, position of the MID; None where the pattern has none)
    patterns = [("111", "SAR aircraft", 3), ("970", "distress device (SART/MOB/EPIRB)", None),
                ("972", "distress device (SART/MOB/EPIRB)", None),
                ("974", "distress device (SART/MOB/EPIRB)", None),
                ("99", "aid to navigation", 2), ("98", "craft of a parent ship", 2),
                ("00", "coast station", 2), ("0", "group of ships", 1), ("8", "handheld", None)]
    patterns += [(d, "ship", 0) for d in "234567"]
    for prefix, kind, at in patterns:
        if s.startswith(prefix):
            if at is not None and int(s[at:at + 3]) not in ITU_MIDS:
                return f"{kind}, unallocated MID"
            return kind
    return "unassigned pattern"


classes = pd.Series({m: mmsi_class(m) for m in ais.mmsi.unique()})
ais["mmsi_class"] = ais.mmsi.map(classes)
mmsi_check = (ais.groupby("mmsi_class")
                 .agg(MMSIs=("mmsi", "nunique"), positions=("mmsi", "size"))
                 .sort_values("positions", ascending=False))
save(mmsi_check.reset_index(), "01_mmsi_check", "ITU-R M.585 classes")
print(mmsi_check.to_string(), "\n")
ais = ais[ais.mmsi_class == "ship"].drop(columns="mmsi_class").reset_index(drop=True)
ais["time"] = pd.to_datetime(ais.unix_time, unit="s")
# AIS "not available" codes -> missing values
ais["speed_over_ground"] = ais.speed_over_ground.where(ais.speed_over_ground < 102.3)

print(f"{len(ais):,} positions · {ais.mmsi.nunique():,} vessels")
print(f"{ais.time.min()} to {ais.time.max()} UTC")
save(ais.vessel_type.value_counts(), "01_ais_vessel_types", "fleet composition")

ais.vessel_type.value_counts().head(6)

  saved  01_mmsi_check.csv                               0.2 KB  ITU-R M.585 classes
                                  MMSIs  positions
mmsi_class                                        
ship                               5036   18089555
placeholder                           2     372629
group of ships                        1       3502
craft of a parent ship                1        311
ship, unallocated MID                 1         50
distress device (SART/MOB/EPIRB)      7         23
coast station, unallocated MID        3          5 



18,089,555 positions · 5,036 vessels
2015-10-01 00:00:00 to 2016-03-31 21:59:59 UTC


  saved  01_ais_vessel_types.csv                         0.4 KB  fleet composition


vessel_type
Passenger       4921385
Fishing         3365383
Unknown         2918153
Cargo           1535399
Tug             1457758
Military Ops     761682
Name: count, dtype: int64

### The study area

Weather stations against the vessel traffic they have to describe. The stations ring the coast; the shipping lanes run well offshore, which is why interpolation &mdash; not lookup &mdash; is needed.

In [5]:
ais_merc = ais.iloc[::max(1, len(ais) // 20_000)].to_crs(3857)   # ~20,000 are plenty
traffic = pd.DataFrame({"mx": ais_merc.geometry.x, "my": ais_merc.geometry.y})

study_area = traffic.hvplot.points(
    x="mx", y="my", color="#0969da", size=3, alpha=0.25,
    tiles="OSM", data_aspect=1, frame_width=620, frame_height=520,
    xaxis=None, yaxis=None, title="AIS traffic and weather stations",
) * weather.drop_duplicates("station").hvplot.points(
    x="mx", y="my", color="#e16f24", size=140, marker="triangle",
    line_color="white", hover_cols=["station"],
)

save(study_area, "01_study_area_map", "traffic and stations on OSM")
study_area

  saved  01_study_area_map.html                        283.9 KB  traffic and stations on OSM


:Overlay
   .Tiles.I   :Tiles   [x,y]
   .Points.I  :Points   [mx,my]
   .Points.II :Points   [mx,my]   (station)

### The resolution gap

One vessel's positions against the weather epochs. Every AIS dot between two weather ticks has no context value of its own &mdash; that is the gap annotation fills.

In [6]:
busiest = ais.mmsi.value_counts().index[0]
one = ais[ais.mmsi == busiest].sort_values("unix_time")
# Plots thin a long track to about 5,000 points so the notebook stays light;
# every computation below uses all of them.
STEP = max(1, len(one) // 5000)
epochs = pd.to_datetime(sorted(weather.unix_time.unique()), unit="s")

gap = pd.concat([
    pd.DataFrame({"time": one.time.iloc[::STEP], "row": "movement", "kind": "AIS position"}),
    pd.DataFrame({"time": epochs, "row": "context", "kind": "weather epoch"}),
])

print(f"{len(one):,} positions against {len(epochs)} context epochs")

gap.hvplot.scatter(
    x="time", y="row", by="kind", size=40, alpha=0.7,
    frame_width=740, frame_height=170, ylabel="",
    title=f"vessel {busiest}",
).opts(legend_position="top_right")

1,168,577 positions against 4387 context epochs


:NdOverlay   [kind]
   :Scatter   [time]   (row)

---
## 2. Stage 1 &mdash; spatial interpolation

Context is measured at stations, never where the moving object is. For each epoch we interpolate the station values **directly to the query position** &mdash; no raster is built.

Take a single epoch and look at what the stations recorded — here **mean wind speed**, the `wind_speed` column (`Ff` in the source). Hover a marker for its identifier and readings.

In [7]:
epoch = sorted(weather.unix_time.unique())[12]   # midday-ish
snap = weather[weather.unix_time == epoch]

station_map = snap.hvplot.points(
    x="mx", y="my", c="wind_speed", cmap="viridis", size=200, line_color="white",
    hover_cols=["station", "air_temperature", "air_pressure"], clabel="wind speed (m/s)",
    tiles="OSM", data_aspect=1, frame_width=560, frame_height=480,
    xaxis=None, yaxis=None,
    title=f"{labels['wind_speed']} at {pd.to_datetime(epoch, unit='s')} UTC",
)

save(snap.drop(columns="geometry"), "02_epoch_snapshot", "the epoch used below")
save(station_map, "02_station_map")
station_map

  saved  02_epoch_snapshot.csv                           2.9 KB  the epoch used below
  saved  02_station_map.html                            23.1 KB  


:Overlay
   .Tiles.I  :Tiles   [x,y]
   .Points.I :Points   [mx,my]   (wind_speed,station,air_temperature,air_pressure)

### The candidate methods

| | |
|---|---|
| **Nearest** | piecewise constant &mdash; an error floor |
| **IDW** | inverse distance weighting, `w = 1/d^p` |
| **TIN** | linear on a Delaunay triangulation |
| **CloughTocher** | cubic C&sup1; on the same triangulation |
| **RBF** | thin-plate spline, smooth |
| **Kriging** | Gaussian process &mdash; also returns uncertainty |

All are *exact* at the stations except Kriging with a nugget: a query landing on a station returns that station's value.

In [8]:
centre = (snap.x.mean(), snap.y.mean())

comparison = pd.DataFrame([
    {"method": M.__name__,
     "estimate_at_centroid": M().fit(snap.x, snap.y, snap.wind_speed)(*centre)}
    for M in (Nearest, IDW, TIN, CloughTocher, RBF, Kriging)
])

fit = IDW().fit(snap.x, snap.y, snap.wind_speed)
print(f"IDW at station 1: {fit(snap.x.iloc[0], snap.y.iloc[0]):.3f}"
      f"  (observed {snap.wind_speed.iloc[0]:.3f})  -- exact at the stations\n")

save(comparison, "02_methods_at_centroid", "one epoch, six methods")
comparison.round(3)

IDW at station 1: 7.000  (observed 7.000)  -- exact at the stations

  saved  02_methods_at_centroid.csv                      0.2 KB  one epoch, six methods


,method,estimate_at_centroid
0,Nearest,5.000
1,IDW,5.655
2,TIN,5.414
3,CloughTocher,5.128
4,RBF,5.036
5,Kriging,5.312


### Surfaces, to see the difference

Annotation samples at points and never needs a raster, but `to_surface` builds one &mdash; useful for seeing how the methods differ. The panels share axes, so zooming one zooms all six.

Watch the triangulation methods: they return **nothing outside the convex hull** of the stations. That is deliberate. They have no basis for a value there, and admitting the gap is better than inventing one.

In [9]:
pad = 40_000
bounds = (snap.x.min() - pad, snap.y.min() - pad,
          snap.x.max() + pad, snap.y.max() + pad)

stations_utm = snap.hvplot.points(
    x="x", y="y", color="white", size=40, line_color="black",
    hover_cols=["station", "wind_speed"],
)

panels = []
for m in [Nearest(), IDW(power=2), TIN(), CloughTocher(), RBF(), Kriging()]:
    grid, xs, ys = to_surface(m.fit(snap.x, snap.y, snap.wind_speed), bounds, 6000)
    name = type(m).__name__
    img = hv.Image((xs, ys, grid), kdims=["x", "y"], vdims="wind speed").opts(
        cmap="viridis", frame_width=250, frame_height=230, colorbar=False,
        xaxis=None, yaxis=None,
        title=f"{name}  ({100 * np.isfinite(grid).mean():.0f}% covered)",
    )
    panels.append(img * stations_utm)
    pd.DataFrame(grid, index=ys, columns=xs).to_csv(
        RESULTS / f"02_surface_{name.lower()}.csv")

surfaces = hv.Layout(panels).cols(3).opts(shared_axes=True)
print("  saved  six interpolated grids as 02_surface_*.csv")
save(surfaces, "02_surfaces", "six interpolators, one epoch")
surfaces

  saved  six interpolated grids as 02_surface_*.csv


  saved  02_surfaces.html                              509.1 KB  six interpolators, one epoch


:Layout
   .Overlay.I   :Overlay
      .Image.I  :Image   [x,y]   (wind speed)
      .Points.I :Points   [x,y]   (station,wind_speed)
   .Overlay.II  :Overlay
      .Image.I  :Image   [x,y]   (wind speed)
      .Points.I :Points   [x,y]   (station,wind_speed)
   .Overlay.III :Overlay
      .Image.I  :Image   [x,y]   (wind speed)
      .Points.I :Points   [x,y]   (station,wind_speed)
   .Overlay.IV  :Overlay
      .Image.I  :Image   [x,y]   (wind speed)
      .Points.I :Points   [x,y]   (station,wind_speed)
   .Overlay.V   :Overlay
      .Image.I  :Image   [x,y]   (wind speed)
      .Points.I :Points   [x,y]   (station,wind_speed)
   .Overlay.VI  :Overlay
      .Image.I  :Image   [x,y]   (wind speed)
      .Points.I :Points   [x,y]   (station,wind_speed)

### Which method is best? Leave-one-station-out

Hold out one station, predict its value from the others, compare against what it actually observed. Repeat for every station, at every epoch.

This is the spatial analogue of k-fold, with **stations rather than time** as the folds.

Two things get reported that a bare error figure would hide:

- **unreachable predictions** &mdash; a held-out station may fall outside the hull of the others. Corner stations often do, and the triangulation methods cannot reach them. A method that fails at edge stations is worse in a way its error alone does not show.
- **station count** &mdash; every fit uses one fewer station than exist. With 16 stations, that is thin for kriging.

In [10]:
table = evaluate_spatial(weather.x, weather.y, weather.wind_speed, weather.unix_time,
                         n_jobs=N_JOBS)   # epochs are independent: scored in parallel
print(table.summary())

df = table.to_frame()

save(table.summary(), "02_spatial_cv_summary")
save(df, "02_spatial_cv_by_epoch", "every method at every epoch")
ranked = (df.groupby("method")[["rmse", "mae", "me", "max_ae", "r2", "unreachable"]]
            .mean().sort_values("rmse"))
save(ranked.reset_index(), "02_spatial_cv_ranked", "averaged over epochs")

ranked.round(3)

leave-one-station-out over 4387 epoch(s), 2-30 stations
  tin            mean rmse=1.811   unreachable=26082
  clough_tocher  mean rmse=2.026   unreachable=26082
  idw            mean rmse=2.552
  kriging        mean rmse=2.582
  rbf            mean rmse=2.72   unreachable=30
  nearest        mean rmse=2.774
  saved  02_spatial_cv_summary.txt                       0.3 KB  


  saved  02_spatial_cv_by_epoch.csv                   2979.1 KB  every method at every epoch
  saved  02_spatial_cv_ranked.csv                        0.7 KB  averaged over epochs


,rmse,mae,me,max_ae,r2,unreachable
method,,,,,,
tin,1.811,1.495,0.009,3.365,0.062,5.945
clough_tocher,2.026,1.682,0.002,3.757,-0.226,5.945
idw,2.552,2.061,0.015,5.515,0.038,0.000
kriging,2.582,2.083,0.014,5.638,-0.009,0.000
rbf,2.720,2.142,-0.069,6.270,-0.272,0.007
nearest,2.774,2.191,0.096,5.847,-0.199,0.000


Five metrics are reported together because no one of them is sufficient:

- **RMSE** and **MAE** are both magnitude-only. A model that consistently under-estimates by 2 m/s scores identically to one scattering &plusmn;2 m/s symmetrically.
- **ME** is signed, so it is the only one that reveals *bias*.
- **Max|e|** exposes rare catastrophic points that RMSE averages away.
- The **gap between RMSE and MAE** is itself diagnostic &mdash; a large gap means a few big errors, a small gap means they are spread evenly.

Click a method in the legend to mute it.

In [11]:
df["time"] = pd.to_datetime(df.epoch, unit="s")

cv_by_epoch = df.hvplot.line(
    x="time", y="rmse", by="method", muted_alpha=0.1,
    frame_width=700, frame_height=300, ylabel="RMSE (m/s)",
    title="leave-one-station-out error, by epoch",
).opts(legend_position="right")

save(cv_by_epoch, "02_spatial_cv_by_epoch_plot")
cv_by_epoch

  saved  02_spatial_cv_by_epoch_plot.html              617.7 KB  


:NdOverlay   [method]
   :Curve   [time]   (rmse)

That variation is why **per-epoch selection** exists as an option. A calm, smooth field and a storm front are not best served by the same interpolator.

It is not the default, though. The method can then switch between consecutive hours, and a value that moves because the *interpolator* changed rather than the weather is an artefact &mdash; one the temporal stage will faithfully reproduce as roughness.

In [12]:
overall = table.select()
per_epoch = table.select(per_epoch=True)

print("one method for all epochs :", overall)
print("\nper-epoch picks:")
print(pd.Series(list(per_epoch.values())).value_counts().to_string())

save(pd.DataFrame({"epoch": list(per_epoch), "method": list(per_epoch.values())})
       .assign(time=lambda d: pd.to_datetime(d.epoch, unit="s")),
     "02_spatial_method_per_epoch", f"the single-method choice was {overall}")

one method for all epochs : tin

per-epoch picks:
tin              3027
clough_tocher     910
idw               221
kriging           167
rbf                49
nearest            11
  saved  02_spatial_method_per_epoch.csv               163.8 KB  the single-method choice was tin


,epoch,method,time
0,1443657600,tin,2015-10-01 00:00:00
1,1443661200,tin,2015-10-01 01:00:00
2,1443664800,tin,2015-10-01 02:00:00
3,1443668400,tin,2015-10-01 03:00:00
4,1443672000,tin,2015-10-01 04:00:00
...,...,...,...
4380,1459458000,tin,2016-03-31 21:00:00
4381,1459461600,tin,2016-03-31 22:00:00
4382,1459465200,tin,2016-03-31 23:00:00
4383,1459468800,tin,2016-04-01 00:00:00


### Wind direction is different &mdash; it is a bearing

0&deg; and 360&deg; are the same direction. Interpolated as ordinary numbers, stations reading 350&deg; and 10&deg; average to **180&deg;** &mdash; the exact opposite heading.

So bearings are interpolated **on the unit circle**: convert each to a vector, interpolate `cos` and `sin` independently, recover with `atan2`.

**The caller declares which variables are circular.** Nothing is inferred from a column name.

This dataset also stores direction as a **compass-point code**, not degrees &mdash; `degrees = code x 22.5` on a 16-point rose. Interpolating the codes would be meaningless; they are categorical labels, and a step from code 5 to code 13 is not a rotation.

In [13]:
# The dictionary is what tells us this column is a *compass point*, not a
# bearing in degrees -- the column name alone would not.
print(labels["wind_direction_code"], "\n")

weather["wind_direction"] = decode_compass(weather.wind_direction_code)
print("codes  :", sorted(weather.wind_direction_code.dropna().unique())[:8])
print("degrees:", sorted(weather.wind_direction.dropna().unique())[:8])

# A worked example that crosses north, where the two treatments diverge.
xs_n = np.array([0.0, 1e5, 0.0])
ys_n = np.array([0.0, 0.0, 1e5])
bearings = np.array([350.0, 10.0, 0.0])
q = (3e4, 3e4)

plain = IDW(circular=False).fit(xs_n, ys_n, bearings)(*q)
circ = IDW(circular=True).fit(xs_n, ys_n, bearings)(*q)
print(f"\nstations reading {bearings.tolist()} degrees:")
print(f"  as plain numbers : {plain:6.1f}   <- points south, wrong")
print(f"  as bearings      : {circ:6.1f}   <- correct")

wind_direction_code - Mean wind direction (compass point) 

codes  : [np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8)]
degrees: [np.float64(0.0), np.float64(22.5), np.float64(45.0), np.float64(67.5), np.float64(90.0), np.float64(112.5), np.float64(135.0), np.float64(157.5)]

stations reading [350.0, 10.0, 0.0] degrees:
  as plain numbers :  217.9   <- points south, wrong
  as bearings      :  355.7   <- correct


---
## 3. Stage 2 &mdash; temporal interpolation

Stage 1 gave us a value at the object's position for every epoch. Now we fit a curve through that series and read it at the exact timestamp.

Three candidates, from [Mehri2023] &sect;4.1:

| | | |
|---|---|---|
| **DLT** | Eq. 2 | straight line between consecutive epochs |
| **PiecewisePolynomial** | Eq. 3 | contiguous least-squares windows, degrees 2&ndash;7 |
| **Spline** | Eq. 4 | one cubic curve through every observation |

DLT assumes the variable changes linearly across the gap. Real environmental variables curve, and the coarser the observations the worse a straight line fits &mdash; which is what motivates the two nonlinear methods.

Overlaying them shows where they actually *disagree*. Zoom into a single gap.

In [14]:
sid = weather.station.iloc[0]
series = weather[weather.station == sid].sort_values("unix_time")
t, v = series.unix_time.to_numpy(float), series.wind_speed.to_numpy(float)
dense = np.linspace(t.min(), t.max(), 600)

fits = {"DLT": DLT(),
        "PiecewisePolynomial (deg 2)": PiecewisePolynomial(degree=2),
        "Spline": Spline()}

curves = pd.concat([
    pd.DataFrame({"time": pd.to_datetime(dense, unit="s"),
                  "wind speed": m.fit(t, v)(dense), "method": name})
    for name, m in fits.items()
])
observed = pd.DataFrame({"time": pd.to_datetime(t, unit="s"), "wind speed": v})

method_curves = curves.hvplot.line(
    x="time", y="wind speed", by="method", muted_alpha=0.15,
    frame_width=700, frame_height=320, ylabel="wind speed (m/s)",
    title=f"station {sid} - the same {len(t)} observations, three methods",
) * observed.hvplot.scatter(
    x="time", y="wind speed", color="#e16f24", size=70, line_color="white",
)

save(curves, "03_temporal_fitted_curves", "three methods, densely sampled")
save(method_curves, "03_temporal_methods_plot")
method_curves

  saved  03_temporal_fitted_curves.csv                 106.7 KB  three methods, densely sampled


  saved  03_temporal_methods_plot.html                 129.7 KB  


:Overlay
   .NdOverlay.I :NdOverlay   [method]
      :Curve   [time]   (wind speed)
   .Scatter.I   :Scatter   [time]   (wind speed)

Zoom in on the piecewise polynomial and you will see its curve **does not pass through the observations** &mdash; it passes *near* them.

That is not a bug. Eq. 3 partitions the interval into contiguous windows, and with `window = degree + 2` each sub-polynomial has `d+1` coefficients fitted to `d+2` points &mdash; a least-squares fit with one degree of freedom. A window of exactly `d+1` points would be forced through every one, leaving zero residual and making every degree score identically.

The spline, by contrast, is **interpolating**: it passes through every observation exactly, and matches slope and curvature at every knot. That difference is what the comparison is really measuring.

In [15]:
residuals = pd.DataFrame([
    {"method": name,
     "max_abs_residual_at_observations": float(np.abs(m.fit(t, v)(t) - v).max())}
    for name, m in [("PiecewisePolynomial", PiecewisePolynomial(degree=2)),
                    ("Spline", Spline())]
])

save(residuals, "03_interpolating_vs_least_squares",
     "zero means the curve passes through every observation")
residuals

  saved  03_interpolating_vs_least_squares.csv           0.1 KB  zero means the curve passes through every observation


,method,max_abs_residual_at_observations
0,PiecewisePolynomial,NaN
1,Spline,NaN


### Which method is best? k-fold

Observations are partitioned **randomly** into k sets. Each fold is held out in turn, the rest fits the model, the held-out fold is scored. Reported metrics are the mean over all k folds.

`k` does double duty: training size grows with k, so sweeping k = 2&hellip;10 varies both the fold structure *and* the number of fitting points.

**The first and last observation are always kept in training.** Random partitioning could otherwise hold one out, leaving the training set unable to span it &mdash; and the model would be extrapolating, not interpolating. Context observations bracket the trajectory by design, so extrapolation is prevented structurally rather than merely scored.

In [16]:
ttable = evaluate_temporal(t, v)
print(ttable.summary())

save(ttable.summary(), "03_temporal_cv_summary")
save(ttable.to_frame(), "03_temporal_cv_full", "every method at every k")

kfold_heatmap = ttable.to_frame().hvplot.heatmap(
    x="k", y="method", C="rmse", cmap="viridis_r",
    frame_width=520, frame_height=300, clabel="RMSE (m/s)",
    title="every method at every fold count - darker is better",
).opts(xrotation=0)

save(kfold_heatmap, "03_temporal_cv_heatmap")
kfold_heatmap

k-fold over k=2..10, 4365 observations
  dlt          best k=10  rmse=1.157   MAE=0.8198  n_fit=3924
  pp_deg2      best k=10  rmse=1.235   MAE=0.8922  n_fit=3924
  pp_deg3      best k=9   rmse=1.377   MAE=0.9919  n_fit=3875
  spline       best k=10  rmse=1.407   MAE=1.019  n_fit=3924
  pp_deg4      best k=10  rmse=1.702   MAE=1.17  n_fit=3924
  pp_deg5      best k=8   rmse=2.354   MAE=1.492  n_fit=3815
  pp_deg6      best k=9   rmse=3.464   MAE=1.865  n_fit=3875
  pp_deg7      best k=9   rmse=6.378   MAE=2.67  n_fit=3875
  saved  03_temporal_cv_summary.txt                      0.5 KB  
  saved  03_temporal_cv_full.csv                         8.2 KB  every method at every k
  saved  03_temporal_cv_heatmap.html                    22.8 KB  


:HeatMap   [k,method]   (rmse)

### Selection is deliberately a separate step

`evaluate_temporal` returns the table; the analyst chooses. The published study shows why that is not purely mechanical &mdash; for mean wind speed, DLT won on MAE while the spline won on RMSE, and the spline was chosen *because* RMSE weights the large errors that matter for this application.

`select()` returns the winner **already fitted to every observation**, ready to annotate with.

In [17]:
by_rmse, by_mae = ttable.best_row("rmse"), ttable.best_row("mae")
print(f"best by RMSE: {by_rmse.method} (k={by_rmse.k})")
print(f"best by MAE : {by_mae.method} (k={by_mae.k})")
if (by_rmse.method, by_rmse.k) != (by_mae.method, by_mae.k):
    print("\n-> they disagree; the choice is a judgement, not a default")

model = ttable.select()          # or ttable.select(method="spline")
print(f"\nselected and fitted: {model}")

save(pd.DataFrame([
    {"rule": "rmse", "method": by_rmse.method, "k": by_rmse.k,
     "score": by_rmse.metrics.rmse},
    {"rule": "mae", "method": by_mae.method, "k": by_mae.k,
     "score": by_mae.metrics.mae},
]), "03_temporal_selection", "the two rules, and what each chose")

best by RMSE: dlt (k=10)
best by MAE : dlt (k=10)

selected and fitted: DLT(circular=False)
  saved  03_temporal_selection.csv                       0.1 KB  the two rules, and what each chose


,rule,method,k,score
0,rmse,dlt,10,1.157331
1,mae,dlt,10,0.819811


---
## 4. Putting it together

`annotate_points` runs both stages: spatial across stations at each epoch, then temporal through the resulting series.

The stations table just needs four columns: `x`, `y`, `t`, `value` — whatever the source called them.

Every position is annotated — no sampling. Each epoch's spatial interpolator is fitted once and evaluated at all positions together, so the whole track costs a few seconds.

In [18]:
stations = pd.DataFrame({
    "x": weather.x, "y": weather.y,
    "t": weather.unix_time.astype(float),
    "value": weather.wind_speed.astype(float),
})

track = one.copy()                     # every point, not a sample
track["wind_speed"] = annotate_points(
    stations,
    track.geometry.x.to_numpy(), track.geometry.y.to_numpy(),
    track.unix_time.to_numpy(float),
    spatial_method=IDW(power=2), temporal_method=Spline(), n_jobs=N_JOBS,
)

print(f"annotated {track.wind_speed.notna().sum()}/{len(track)} positions "
      f"of vessel {busiest}")
track[["time", "speed_over_ground", "wind_speed"]].head()

annotated 1168577/1168577 positions of vessel 228186700


,time,speed_over_ground,wind_speed
11993237,2015-10-01 00:00:34,0.0,7.051403
11993238,2015-10-01 00:03:34,0.0,7.030704
11993239,2015-10-01 00:06:34,0.0,7.010691
11993240,2015-10-01 00:09:34,0.0,6.991172
11993241,2015-10-01 00:12:37,0.0,6.971638


The track on a basemap, coloured by the context annotation gave it. Hover a position to read the vessel's own speed alongside the wind speed it was moving through.

In [19]:
track_merc = track.to_crs(3857)
track_plot = track.assign(
    mx=track_merc.geometry.x, my=track_merc.geometry.y,
    hour=track.time.dt.strftime("%H:%M"),
).iloc[::STEP]

track_map = track_plot.hvplot.points(
    x="mx", y="my", c="wind_speed", cmap="viridis", size=30,
    hover_cols=["hour", "speed_over_ground"], clabel="wind speed (m/s)",
    tiles="OSM", data_aspect=1, frame_width=560, frame_height=460,
    xaxis=None, yaxis=None, title=f"vessel {busiest}, annotated",
) * snap.hvplot.points(
    x="mx", y="my", color="#e16f24", marker="triangle", size=120,
    line_color="white", hover_cols=["station"],
)

save(track_map, "04_annotated_track_map", "on a basemap")
track_map

  saved  04_annotated_track_map.html                   191.0 KB  on a basemap


:Overlay
   .Tiles.I   :Tiles   [x,y]
   .Points.I  :Points   [mx,my]   (wind_speed,hour,speed_over_ground)
   .Points.II :Points   [mx,my]   (station)

The annotated series is smooth and follows the regional trend, but it is **not** the station mean &mdash; it tracks conditions along *this vessel's* path, which is the whole point.

In [20]:
station_mean = weather.groupby("time").wind_speed.mean().rename("station mean").reset_index()

along_path = track.iloc[::STEP].hvplot.line(
    x="time", y="wind_speed", color="#1a7f37", line_width=2,
    frame_width=700, frame_height=280, ylabel="wind speed (m/s)",
    label="annotated along the track", title="context along the trajectory",
) * station_mean.hvplot.scatter(
    x="time", y="station mean", color="#e16f24", size=45,
    label=f"mean of all {weather.station.nunique()} stations",
)

save(along_path, "04_context_along_path")
along_path

  saved  04_context_along_path.html                    350.8 KB  


:Overlay
   .Curve.Annotated_along_the_track :Curve   [time]   (wind_speed)
   .Scatter.Mean_of_all_16_stations :Scatter   [time]   (station mean)

### With MovingPandas

`annotate` writes onto a trajectory's dataframe. Because context lives as an ordinary column, it survives the `.iloc` slicing that trajectory compression performs &mdash; so a compressed trajectory keeps the context of the points it retains.

In [21]:
import movingpandas as mpd

traj = mpd.Trajectory(track.set_index("time"), traj_id=str(busiest))
annotate(traj, stations, "wind_speed_mpd",
         spatial_method=IDW(power=2), temporal_method=Spline(), n_jobs=N_JOBS)

print(traj)
traj.df[["speed_over_ground", "wind_speed_mpd"]].head()

Trajectory 228186700 (2015-10-01 00:00:34 to 2016-03-31 21:59:54) | Size: 1168577 | Length: 4733.1 km
Bounds: (220927.9024659503, 5259941.846699245, 399200.51300929027, 5381731.066014169)
LINESTRING (387974.5426612741 5358623.678459492, 387974.5426612741 5358623.678459492, 387974.5426612


,speed_over_ground,wind_speed_mpd
time,,
2015-10-01 00:00:34,0.0,7.051403
2015-10-01 00:03:34,0.0,7.030704
2015-10-01 00:06:34,0.0,7.010691
2015-10-01 00:09:34,0.0,6.991172
2015-10-01 00:12:37,0.0,6.971638


---
## 5. Several variables at once

Each context variable gets its own choice of method &mdash; and `circular=True` where the variable is a bearing.

The panels share a time axis, so zooming one zooms them all: a wind shift can be read against the pressure change that accompanied it.

In [22]:
variables = {
    "wind_speed":      ("wind_speed", False, "m/s"),
    "air_temperature": ("air_temperature", False, "degC"),
    "air_pressure":    ("air_pressure", False, "mmHg"),
    "wind_direction":  ("wind_direction", True, "degrees"),   # a bearing
}

for name, (col, circular, _) in variables.items():
    st = stations.assign(value=weather[col].astype(float).to_numpy())
    track[name] = annotate_points(
        st, track.geometry.x.to_numpy(), track.geometry.y.to_numpy(),
        track.unix_time.to_numpy(float),
        spatial_method=IDW(power=2, circular=circular),
        temporal_method=Spline(circular=circular),
        circular=circular, n_jobs=N_JOBS,
    )

save(track.drop(columns="geometry"), "04_annotated_track_all_variables",
     "four context variables on one track")
save(gpd.GeoDataFrame(track, geometry="geometry", crs=ais.crs),
     "04_annotated_track", "same, with geometry -- opens in QGIS")

# `wind_direction` is a column we derived, so it is described by the source
# column it came from.
source_of = {"wind_direction": "wind_direction_code"}

print("annotated:")
for name, (col, _, unit) in variables.items():
    described = labels.get(source_of.get(col, col), col)
    print(f"  {name:12} {described}")

track[["time"] + list(variables)].head()

  saved  04_annotated_track_all_variables.csv       198051.4 KB  four context variables on one track


  saved  04_annotated_track.gpkg                    236352.0 KB  same, with geometry -- opens in QGIS
annotated:
  wind_speed   wind_speed - Mean wind speed (meters per seconds)
  air_temperature air_temperature - Air temperature (°C)
  air_pressure air_pressure - Atmospheric pressure (mm of mercury)
  wind_direction wind_direction_code - Mean wind direction (compass point)


,time,wind_speed,air_temperature,air_pressure,wind_direction
11993237,2015-10-01 00:00:34,7.051403,13.474055,770.385586,108.755071
11993238,2015-10-01 00:03:34,7.030704,13.471314,770.383183,108.677870
11993239,2015-10-01 00:06:34,7.010691,13.468451,770.380040,108.607590
11993240,2015-10-01 00:09:34,6.991172,13.465400,770.376237,108.543785
11993241,2015-10-01 00:12:37,6.971638,13.462039,770.371777,108.485094


In [23]:
panels_all = hv.Layout([
    track.iloc[::STEP].hvplot.line(
        x="time", y=name, color="#0969da",
        frame_width=680, frame_height=150, ylabel=f"{name} ({unit})",
    )
    for name, (_, _, unit) in variables.items()
]).cols(1).opts(shared_axes=True)

save(panels_all, "04_all_variables_plot", "linked time axes")
panels_all

  saved  04_all_variables_plot.html                   1183.3 KB  linked time axes


:Layout
   .Curve.I   :Curve   [time]   (wind_speed)
   .Curve.II  :Curve   [time]   (air_temperature)
   .Curve.III :Curve   [time]   (air_pressure)
   .Curve.IV  :Curve   [time]   (wind_direction)

---
## 6. Every vessel

Sections 4 and 5 annotated one trajectory. `annotate_points` takes any number of positions, so the same call annotates the whole fleet, every position in the data.

Two choices for a run this size:

* **Spatial: IDW.** TIN may score best in the cross-validation, but it returns nothing outside the hull of the stations, and much of the traffic lies offshore of it. IDW reaches every position.
* **Temporal: DLT**, a straight line between the hourly epochs. Compare it with the k-fold result in section 3. It is evaluated for all positions at once, so millions of positions take minutes rather than hours.

Long inputs are processed in time blocks automatically (`chunk_epochs`), so memory stays bounded however long the period, and the blocks run in parallel on `N_JOBS` processes. The full annotated table is written to the results folder as a Parquet file (it needs `pyarrow`); it is large, about 1 GB for the whole archive, so set `EXPORT_FLEET = False` to skip it.

In [24]:
import time as clock

EXPORT_FLEET = True           # writes every annotated position to a Parquet file (~1 GB for the archive)

fleet = pd.DataFrame({"mmsi": ais.mmsi.to_numpy(), "vessel_type": ais.vessel_type.to_numpy(),
                      "time": ais.time.to_numpy(),
                      "speed_over_ground": ais.speed_over_ground.to_numpy()})
fx, fy, ft = ais.geometry.x.to_numpy(), ais.geometry.y.to_numpy(), ais.unix_time.to_numpy(float)

print(f"annotating {len(fleet):,} positions of {fleet.mmsi.nunique():,} vessels")
for name, (col, circular, unit) in variables.items():
    started = clock.perf_counter()
    fleet[name] = annotate_points(
        stations.assign(value=weather[col].astype(float).to_numpy()), fx, fy, ft,
        spatial_method=IDW(power=2, circular=circular),
        temporal_method=DLT(circular=circular), circular=circular, n_jobs=N_JOBS)
    print(f"  {name:<16} {fleet[name].notna().mean():8.2%} annotated  "
          f"({clock.perf_counter() - started:5.0f} s)")

# Per-vessel means; a bearing's mean is taken on the circle.
groups = fleet.groupby(["mmsi", "vessel_type"])
per_vessel = groups.size().rename("positions").to_frame()
for name, (_, circular, _) in variables.items():
    if circular:
        rad = np.radians(fleet[name])
        s = rad.apply(np.sin).groupby([fleet.mmsi, fleet.vessel_type]).mean()
        c = rad.apply(np.cos).groupby([fleet.mmsi, fleet.vessel_type]).mean()
        per_vessel[name] = np.mod(np.degrees(np.arctan2(s, c)), 360)
    else:
        per_vessel[name] = groups[name].mean()
per_vessel = per_vessel.round(2).reset_index().sort_values("positions", ascending=False)
save(per_vessel, "06_fleet_context_per_vessel", "mean context per vessel")
if EXPORT_FLEET:
    # Parquet, not CSV: millions of rows compress to a fraction of the size and
    # load in seconds (pandas.read_parquet, QGIS, DuckDB ...). Needs pyarrow.
    path = RESULTS / "06_fleet_annotated.parquet"
    fleet.assign(x=fx, y=fy).to_parquet(path, index=False)
    print(f"  saved  {path.name:<42} {path.stat().st_size / 2**20:8.1f} MB  "
          f"every position, every variable (x, y in UTM 30N)")
per_vessel.head(10)

annotating 18,089,555 positions of 5,036 vessels


  wind_speed        100.00% annotated  (   27 s)


  air_temperature   100.00% annotated  (   26 s)


  air_pressure      100.00% annotated  (   26 s)


  wind_direction    100.00% annotated  (   52 s)


  saved  06_fleet_context_per_vessel.csv               225.4 KB  mean context per vessel


,mmsi,vessel_type,positions,wind_speed,air_temperature,air_pressure,wind_direction
1106,228186700,Unknown,1168577,7.93,9.89,763.19,220.77
817,227574020,Passenger,1047973,8.12,10.52,762.09,251.27
910,227705102,Passenger,1016442,8.26,11.09,762.31,246.61
1037,228051000,Tug,957436,8.06,11.24,762.93,244.93
825,227580520,Passenger,784629,8.00,10.54,763.55,254.08
451,227003050,Passenger,718914,8.72,10.18,761.90,264.86
1255,228762000,Military Ops,495073,8.50,8.88,761.77,281.10
865,227635210,Passenger,394077,7.92,10.70,762.30,270.02
1003,228017700,Passenger,377710,8.91,11.55,762.04,237.20
470,227008170,Unknown,327892,8.57,9.99,762.04,270.76


In [25]:
step = max(1, len(fleet) // 30_000)          # a map needs a sample, not millions
shown = ais.iloc[::step].to_crs(3857)
fleet_map = fleet.iloc[::step].assign(
    mx=shown.geometry.x.to_numpy(), my=shown.geometry.y.to_numpy(),
    when=fleet.time.iloc[::step].dt.strftime("%Y-%m-%d %H:%M"),
).hvplot.points(
    x="mx", y="my", c="wind_speed", cmap="viridis", size=4, alpha=0.6,
    hover_cols=["mmsi", "vessel_type", "when", "speed_over_ground"],
    clabel="wind speed (m/s)", tiles="OSM", data_aspect=1,
    frame_width=620, frame_height=520, xaxis=None, yaxis=None,
    title=f"every vessel: wind speed met along the way ({len(fleet) // step:,} of {len(fleet):,} positions shown)",
)
save(fleet_map, "06_fleet_wind_map", "a sample of the annotated fleet")
fleet_map

  saved  06_fleet_wind_map.html                       1919.9 KB  a sample of the annotated fleet


:Overlay
   .Tiles.I  :Tiles   [x,y]
   .Points.I :Points   [mx,my]   (wind_speed,mmsi,vessel_type,when,speed_over_ground)

---

## What was saved

Everything above is now on disk in `results/01_trajectory_annotation/`, so the analysis
survives the kernel. CSVs open anywhere, the GeoPackage opens in QGIS, and the HTML
figures stay interactive in a browser with no kernel and no network.


In [26]:
manifest = pd.DataFrame(
    [{"file": f.name, "kind": f.suffix.lstrip("."),
      "size_kb": round(f.stat().st_size / 1024, 1)}
     for f in sorted(RESULTS.iterdir()) if f.is_file()]
)
manifest.to_csv(RESULTS / "00_manifest.csv", index=False)

print(f"{len(manifest)} files, {manifest.size_kb.sum() / 1024:.1f} MB "
      f"in {RESULTS.name}/")
manifest

34 files, 445.1 MB in results_trajectory_annotation/


,file,kind,size_kb
0,01_ais_vessel_types.csv,csv,0.4
1,01_column_names.csv,csv,1.7
2,01_mmsi_check.csv,csv,0.2
3,01_study_area_map.html,html,283.9
4,01_weather_observations.csv,csv,11798.9
5,02_epoch_snapshot.csv,csv,2.9
6,02_methods_at_centroid.csv,csv,0.2
7,02_spatial_cv_by_epoch.csv,csv,2979.1
8,02_spatial_cv_by_epoch_plot.html,html,617.7
9,02_spatial_cv_ranked.csv,csv,0.7
